# PRISM gemma4: improved pipeline
This notebook uses the modified branch and its own Drive workspace. Use a fresh GPU runtime for each model. Software checks do not establish pretrained accuracy or T4 memory fit. Supply the same shared data/splits for all model comparisons. Setup errors stop the run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
workdir = '/content/drive/MyDrive/prism_ladder_gemma4_modified'
if not os.path.exists(workdir):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'codex/gemma4-model-modified', 'https://github.com/Sanjana432/Samsung-prism-san.git', workdir], check=True)
os.chdir(workdir)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

In [ ]:
# Native Gemma4 vision budget; separate checkpoints and predictions from other models.
import torch, yaml
assert torch.cuda.is_available(), 'Choose a GPU runtime before running model stages.'
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)
assert cfg['model']['base'] == 'google/gemma-4-E2B-it'
if mem < 20:
    cfg['inference']['batch_size'] = 1
    with open('configs/colab_runtime.yaml', 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
print(f'{mem:.0f} GB -> soft token budget', cfg['model']['max_soft_tokens'], 'batch', cfg['inference']['batch_size'])
print('Run the GPU smoke test before long jobs; full Gemma4 E2B GPU memory is not just 2B parameters.')
runtime_config = 'configs/colab_runtime.yaml'
with open(runtime_config, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
def run(script, *args):
    subprocess.run([sys.executable, 'scripts/' + script, '--config', runtime_config, *args], check=True)


In [ ]:
run('run_pipeline.py', '--stages', 'data', 'check')

In [ ]:
run('gpu_smoke_test.py')

In [ ]:
run('03_prompt_study_and_baseline.py')

In [ ]:
run('04_train_qlora.py')

In [ ]:
run('05_infer.py', '--adapter', cfg['paths']['outputs_dir'] + '/best_adapter', '--name', 'finetuned_qlora', '--split', 'test')
run('06_evaluate.py', '--models', 'baseline_zero_shot', 'finetuned_qlora', '--split', 'test')

In [ ]:
from IPython.display import Markdown, Image, display
from pathlib import Path
reports = Path(cfg['paths']['reports_dir'])
display(Markdown((reports / 'evaluation_report.md').read_text()))
display(Image(str(reports / 'figures' / 'mae_per_metric.png')))